# 04 - Feature Engineering

This notebook creates the final feature matrix for modeling by:
- Loading the merged analysis data from EDA
- Pulling additional data if needed (hustle stats, etc.)
- Creating derived features
- Performing feature correlation analysis and selection
- Creating the target variable (`games_missed_next_season`)
- Handling missing values
- Splitting into train/test sets

**Input:** `data/processed/analysis_merged.csv`, `data/processed/team_back_to_backs.csv`

**Output:** `data/final/model_data.csv`, `data/final/train.csv`, `data/final/test.csv`

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.config import (
    FIRST_SEASON, LAST_SEASON,
    PROCESSED_DIR, FINAL_DIR,
    TARGET_COL, RANDOM_SEED
)

# Ensure output directory exists
Path(f'../{FINAL_DIR}').mkdir(parents=True, exist_ok=True)

print(f"Season range: {FIRST_SEASON} to {LAST_SEASON}")
print(f"Target variable: {TARGET_COL}")
print(f"Output directory: {FINAL_DIR}")

---
## 1. Load Processed Data

In [ ]:
# Load merged analysis data from EDA
df = pd.read_csv(f'../{PROCESSED_DIR}/analysis_merged.csv')
print(f"Loaded analysis data: {df.shape}")

# Load back-to-back data
df_b2b = pd.read_csv(f'../{PROCESSED_DIR}/team_back_to_backs.csv')
print(f"Loaded B2B data: {df_b2b.shape}")

print(f"\nSeasons: {sorted(df['season'].unique())}")
print(f"Players: {df['player_id'].nunique()}")

---
## 2. Additional Data Collection (Hustle Stats, etc.)

Hustle stats (contested shots, deflections, charges drawn) showed promising correlations (r=0.19-0.21) in preliminary analysis, but are only available from 2016-17 onward.

**Decision:** Evaluate whether the coverage trade-off is worth it.

In [ ]:
# TODO: Pull hustle stats from nba_api if coverage is sufficient
# from nba_api.stats.endpoints import leaguehustlestatsplayer
#
# Hustle stats available: 2016-17 onward (4 seasons out of 10)
# Coverage: ~40% of our dataset
#
# Decision: DROPPED to maintain full 2010-2019 dataset coverage.
# Can revisit if modeling on 2016+ subset.

print("Hustle stats: SKIPPED (only available 2016+, would reduce dataset to 40% coverage)")
print("Tracking stats: Already merged in EDA (2013+ only, available in analysis_merged.csv)")

---
## 3. Create Derived Features

In [ ]:
# 3a. Age x Minutes interaction term
# Hypothesis: older players with heavy workloads are at highest injury risk
# df['age_x_min'] = df['age'] * df['min']

In [ ]:
# 3b. Prior season injury history (lag feature)
# df = df.sort_values(['player_id', 'season_start_year'])
# df['games_missed_last_season'] = df.groupby('player_id')['games_missed'].shift(1)

In [ ]:
# 3c. Merge B2B schedule data (team-season level)
# df = df.merge(
#     df_b2b[['team_abbreviation', 'season', 'b2b_games', 'b2b_percentage']],
#     left_on=['team_abbreviation', 'season'],
#     right_on=['team_abbreviation', 'season'],
#     how='left'
# )

In [ ]:
# 3d. Position group derived from height
# Guards: <76", Forwards: 76-81", Centers: >81"
# df['position_group'] = pd.cut(
#     df['player_height_inches'].fillna(df['player_height_inches'].median()),
#     bins=[0, 76, 81, 100],
#     labels=['Guard', 'Forward', 'Center']
# )

---
## 4. Feature Correlation Analysis

Examine how derived features correlate with the target and with each other to check for multicollinearity.

In [ ]:
# TODO: Correlation of all candidate features with games_missed
# candidate_features = ['age', 'min', 'age_x_min', 'games_missed_last_season',
#                        'b2b_games', 'b2b_percentage', 'pts', 'tov', 'stl', 'ast',
#                        'dist_miles', 'avg_speed', 'player_weight', 'player_height_inches']
#
# corr_with_target = df[candidate_features + ['games_missed']].corr()['games_missed'].drop('games_missed')
# print(corr_with_target.sort_values(key=abs, ascending=False))

In [ ]:
# TODO: Check multicollinearity between selected features
# corr_matrix = df[candidate_features].corr()
# sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
# plt.title('Feature Correlation Matrix')
# plt.tight_layout()
# plt.show()

---
## 5. Feature Selection (with Rationale)

Based on correlation analysis and data coverage, select the final feature set.

### Selected Features

| Feature | Correlation | Coverage | Rationale |
|---------|-------------|----------|-----------|
| `age_x_min` | r=0.235 | 100% | Interaction: older players + heavy workload = highest risk |
| `min` | r=0.220 | 100% | Workload proxy — more minutes = more exposure |
| `b2b_games` | r=0.216 | 100% | Schedule intensity — more B2Bs = more fatigue |
| `games_missed_last_season` | r=0.213 | 74% | Prior injury is best predictor of future injury |
| `b2b_percentage` | r=0.206 | 100% | Alternative schedule measure |
| `age` | r=0.094 | 100% | Older players have higher baseline risk |
| `position_group` | categorical | 100% | Guard/Forward/Center (derived from height) |

### Features Considered But Dropped

| Feature | Correlation | Why Dropped |
|---------|-------------|-------------|
| Hustle stats (contested_shots, deflections) | r=0.19-0.21 | Only available 2016+ (40% coverage) |
| Tracking (dist_miles, avg_speed) | r=0.17, -0.06 | Only available 2013+ (72% coverage) |
| pts, tov, stl, ast | r=0.15-0.21 | Highly collinear with minutes |

In [ ]:
FEATURES = [
    'age',
    'min',
    'age_x_min',
    'games_missed_last_season',
    'b2b_games',
    'b2b_percentage',
    'position_group'
]

TARGET = 'games_missed_next_season'

print(f"Selected {len(FEATURES)} features + 1 target")
print(f"Features: {FEATURES}")
print(f"Target: {TARGET}")

---
## 6. Create Target Variable

The target is `games_missed_next_season` — the number of games a player will miss in the **following** season. This is created by shifting `games_missed` forward by one season per player.

In [ ]:
# TODO: Create target variable
# df['games_missed_next_season'] = df.groupby('player_id')['games_missed'].shift(-1)
#
# # Drop rows where target is NaN (last season for each player)
# print(f"Before dropping NaN target: {len(df)}")
# df = df.dropna(subset=['games_missed_next_season'])
# print(f"After dropping NaN target: {len(df)}")

---
## 7. Handle Missing Values (Rookies, etc.)

Key missing value scenarios:
- **`games_missed_last_season`**: NaN for rookies (first season in dataset) — fill with 0
- **`position_group`**: NaN if height data missing — fill with mode or impute
- **`b2b_games` / `b2b_percentage`**: Should be complete after merge

In [ ]:
# TODO: Handle missing values
# print("Missing values before handling:")
# print(df[FEATURES + [TARGET]].isnull().sum())
#
# # Rookies: fill games_missed_last_season with 0
# df['games_missed_last_season'] = df['games_missed_last_season'].fillna(0)
#
# print("\nMissing values after handling:")
# print(df[FEATURES + [TARGET]].isnull().sum())

---
## 8. Train/Test Split

Temporal split to prevent data leakage:
- **Train:** 2010-2017 seasons
- **Test:** 2018-2019 seasons

In [ ]:
TRAIN_END_YEAR = 2017

# TODO: Create final dataframe and split
# df_final = df[['player_id', 'player_name', 'season', 'season_start_year'] + FEATURES + [TARGET]]
#
# df_train = df_final[df_final['season_start_year'] <= TRAIN_END_YEAR]
# df_test = df_final[df_final['season_start_year'] > TRAIN_END_YEAR]
#
# print(f"Train: {df_train.shape} (seasons {df_train['season'].min()} to {df_train['season'].max()})")
# print(f"Test:  {df_test.shape} (seasons {df_test['season'].min()} to {df_test['season'].max()})")
# print(f"\nNo overlap: {len(set(df_train['season']) & set(df_test['season'])) == 0}")

---
## 9. Save Final Dataset

In [ ]:
# TODO: Save output files
# df_final.to_csv(f'../{FINAL_DIR}/model_data.csv', index=False)
# df_train.to_csv(f'../{FINAL_DIR}/train.csv', index=False)
# df_test.to_csv(f'../{FINAL_DIR}/test.csv', index=False)
#
# print(f"Saved: {FINAL_DIR}/model_data.csv ({df_final.shape})")
# print(f"Saved: {FINAL_DIR}/train.csv ({df_train.shape})")
# print(f"Saved: {FINAL_DIR}/test.csv ({df_test.shape})")

---
## 10. Handoff Summary for Modeling

### Output Files (`data/final/`)

| File | Rows | Description |
|------|------|-------------|
| `model_data.csv` | TBD | Complete feature matrix with target |
| `train.csv` | TBD | Training set (2010-2017) |
| `test.csv` | TBD | Test set (2018-2019) |

### Final Feature Summary

| Feature | Type | Correlation with Target |
|---------|------|------------------------|
| `age_x_min` | numeric | r=0.235 |
| `min` | numeric | r=0.220 |
| `b2b_games` | numeric | r=0.216 |
| `games_missed_last_season` | numeric | r=0.213 |
| `b2b_percentage` | numeric | r=0.206 |
| `age` | numeric | r=0.094 |
| `position_group` | categorical | — |

### Target Variable

- `games_missed_next_season` (regression target)
- Highly right-skewed — consider log transform
- ~48% zero-inflation — consider zero-inflated models

### Modeling Notes

- Prior injury (`games_missed_last_season`) is the strongest behavioral predictor
- `age_x_min` interaction captures the compounding risk of age + workload
- Position group allows model to capture position-specific injury patterns
- Temporal train/test split prevents data leakage

### Next Steps

1. `05_supervised_models.ipynb` — Train Linear Regression, Random Forest, XGBoost
2. `06_unsupervised_models.ipynb` — K-Means clustering for injury risk profiles